# Makemore: Backpropagation (Becoming a Backprop Ninja)

Following Karpathy's *"Building makemore Part 4: Becoming a Backprop Ninja"* (Zero to Hero, video 5). Same MLP architecture as the activations video, but now we manually derive and verify the gradient for every intermediate variable in the forward pass instead of relying on `loss.backward()`. Then we collapse cross-entropy and batchnorm into single-equation backward forms (what real frameworks do internally), and finally train the network end-to-end using *only* the manual gradients with autograd disabled. The point is to understand what `.backward()` is actually computing under the hood.

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
%matplotlib inline
plt.style.use('dark_background')

In [2]:
# Read in all of the words
words = open("../data/names.txt", 'r').read().splitlines()
print(f'First 5 Words: {words[:5]}, length: {len(words)}')

First 5 Words: ['emma', 'olivia', 'ava', 'isabella', 'sophia'], length: 32033


In [3]:
# Build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(f'itos: {itos}')
print(f'Vocab Size: {vocab_size}')

itos: {1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
Vocab Size: 27


In [4]:
# Function to build a dataset given a list of words

def build_dataset(words):
    block_size = 3 # Context Size
    X, Y = [], []
    
    for word in words:
        context = [0] * block_size

        for c in word + '.':
            ix = stoi[c]
            X.append(context)
            Y.append(ix)

            context = context[1:] + [ix] # Crop and append

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(f'{X.shape=}, {Y.shape=}')
    return X, Y

In [5]:
# Create the trainings splits
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

X.shape=torch.Size([182625, 3]), Y.shape=torch.Size([182625])
X.shape=torch.Size([22655, 3]), Y.shape=torch.Size([22655])
X.shape=torch.Size([22866, 3]), Y.shape=torch.Size([22866])


In [6]:
# Utility function to compare manual gradients against PyTorch Gradients
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | Exact: {str(ex):5s} | Approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [7]:
n_embed = 10        # Dimensionality of the character embedding vectors
n_hidden = 200      # Number of neurons in the hidden layer of MLP
block_size = 3      # Context Length (# chars used to predict next)
seed = 2147483647   # For reproducibility between runs

print(f"Setup: block_size={block_size}, n_embed={n_embed}, n_hidden={n_hidden}")

Setup: block_size=3, n_embed=10, n_hidden=200


In [8]:
# Initialize parameters from the given seed
gen = torch.Generator().manual_seed(seed)                   # Reproducibility
kaiming = (5 / 3) / ((n_embed * block_size)**0.5)           # Uniformity
lookup = torch.randn((vocab_size, n_embed),                 generator=gen)
weights1 = torch.randn((n_embed * block_size, n_hidden),    generator=gen) * kaiming
biases1 = torch.randn(n_hidden,                             generator=gen) * .1
weights2 = torch.randn((n_hidden, vocab_size),              generator=gen) * .1
biases2 = torch.randn(vocab_size,                           generator=gen) * .1

bngain = torch.randn((1, n_hidden)) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden)) * 0.1


parameters = [lookup, weights1, biases1, weights2, biases2, bngain, bnbias]
print(f'Total Parameters: {sum(p.nelement() for p in parameters)}')

for p in parameters:
    p.requires_grad = True 

Total Parameters: 12297


In [ ]:
# A single mini-batch for the gradient-checking exercise below.
# We'll forward-pass on this batch, manually derive every gradient, and compare against autograd's.
batch_size = 32
ix = torch.randint(0, Xtr.shape[0], (batch_size, ), generator=gen)
Xb, Yb = Xtr[ix], Ytr[ix]

In [28]:
# Forward Pass
embed = lookup[Xb]
embedcat = embed.view(embed.shape[0], -1)

# Linear Layer 1
hidden_prebn = embedcat @ weights1 + biases1

# BatchNorm Layer
bnmeani = 1 / batch_size * hidden_prebn.sum(0, keepdim=True)
bndiff = hidden_prebn - bnmeani
bndiff2 = bndiff ** 2
bnvar = 1 / (batch_size - 1) * (bndiff2).sum(0, keepdim=True) # Bessel's correction (dividing by n - 1, not n)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hidden_preact = bngain * bnraw + bnbias

# Non-linearity
hidden = torch.tanh(hidden_preact)

# Linear Layer 2
logits = hidden @ weights2 + biases2

# Cross entropy loss - F.cross_entropy(logits, Yb)
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # Subtract max for numeric stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1 # floating point issue with 1.0 / counts_sum
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(batch_size), Yb].mean()

# PyTorch backward pass
for parameter in parameters:
    parameter.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits, hidden, hidden_preact,
          bnraw, bnvar_inv, bnvar, bndiff, bndiff2, hidden_prebn,
          bnmeani, embedcat, embed]:
    t.retain_grad()
loss.backward()
print(f'{loss.item()=}')

loss.item()=3.8025572299957275


In [ ]:
# Backprop through the whole network manually.
# For every intermediate variable in the forward pass above, derive dL/d<variable> by hand using
# the chain rule, then compare against PyTorch's autograd via cmp().
# When a variable feeds multiple downstream nodes (e.g. bndiff feeds both bndiff2 and bnraw),
# the gradient is the SUM of contributions from each path.

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(batch_size), Yb] = -1.0 / batch_size
dprobs = (1.0 / probs) * dlogprobs
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
dcounts = counts_sum_inv * dprobs
dcounts_sum = (-counts_sum ** -2) * dcounts_sum_inv
dcounts += torch.ones_like(counts) * dcounts_sum
dnorm_logits = (counts) * dcounts
dlogits = dnorm_logits.clone()
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
dhidden = dlogits @ weights2.T
dweights2 = hidden.T @ dlogits
dbiases2 = dlogits.sum(0)
dhidden_preact = (1.0 - hidden ** 2) * dhidden
dbngain = (bnraw * dhidden_preact).sum(0, keepdim=True)
dbnraw = bngain * dhidden_preact
dbnbias = dhidden_preact.sum(0, keepdim=True)
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
dbndiff = bnvar_inv * dbnraw
dbnvar = (-0.5 * (bnvar + 1e-5) ** -1.5) * dbnvar_inv
dbndiff2 = (1.0 / (batch_size - 1)) * torch.ones_like(bndiff2) * dbnvar
dbndiff += (2 * bndiff) * dbndiff2
dhidden_prebn = dbndiff.clone()
dbnmeani = (-dbndiff).sum(0)
dhidden_prebn += 1 / batch_size * (torch.ones_like(hidden_prebn) * dbnmeani)
dembedcat = dhidden_prebn @ weights1.T
dweights1 = embedcat.T @ dhidden_prebn
dbiases1 = dhidden_prebn.sum(0)
dembed = dembedcat.view(embed.shape)
dlookup = torch.zeros_like(lookup)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k, j]
        dlookup[ix] += dembed[k, j]

# Compare every manual gradient against PyTorch's autograd.
# Exact: True means bit-identical. Approximate: True with maxdiff ~1e-9 means correct
# up to floating-point noise (the order of operations differs slightly from autograd's).
cmp('logprobs: ', dlogprobs, logprobs)
cmp('probs: ', dprobs, probs)
cmp('counts_sum_inv: ', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum: ', dcounts_sum, counts_sum)
cmp('counts: ', dcounts, counts)
cmp('norm_logits: ', dnorm_logits, norm_logits)
cmp('logit_maxes: ', dlogit_maxes, logit_maxes)
cmp('logits: ', dlogits, logits)
cmp('hidden: ', dhidden, hidden)
cmp('weights2: ', dweights2, weights2)
cmp('biases2: ', dbiases2, biases2)
cmp('hidden_preact: ', dhidden_preact, hidden_preact)
cmp('bngain: ', dbngain, bngain)
cmp('bnbias: ', dbnbias, bnbias)
cmp('bnraw: ', dbnraw, bnraw)
cmp('bnvar_inv: ', dbnvar_inv, bnvar_inv)
cmp('bnvar: ', dbnvar, bnvar)
cmp('bndiff2: ', dbndiff2, bndiff2)
cmp('bndiff: ', dbndiff, bndiff)
cmp('bnmeani: ', dbnmeani, bnmeani)
cmp('hidden_prebn: ', dhidden_prebn, hidden_prebn)
cmp('embedcat: ', dembedcat, embedcat)
cmp('weights1: ', dweights1, weights1)
cmp('biases1: ', dbiases1, biases1)
cmp('embed: ', dembed, embed)
cmp('Lookup: ', dlookup, lookup)

In [ ]:
# Backprop through cross-entropy in one go.
# Instead of deriving each step (logprobs -> probs -> counts -> norm_logits -> logits) separately,
# the algebra collapses to a single elegant expression:
#     dlogits = softmax(logits) - one_hot(targets), divided by batch size.
# This is what real frameworks compute internally for cross-entropy backward.

loss_fast = F.cross_entropy(logits, Yb)
print(f'Loss: {loss_fast.item()}, Diff: {(loss_fast - loss).item()}')

dlogits = F.softmax(logits, 1)
dlogits[range(batch_size), Yb] -= 1
dlogits /= batch_size

cmp('logits: ', dlogits, logits)

In [ ]:
# Backprop through batch normalization in one go.
# Same idea as the cross-entropy collapse above: instead of going through bnmeani -> bndiff -> bndiff2 ->
# bnvar -> bnvar_inv -> bnraw step by step, we derive a single expression for dhidden_prebn.
# The resulting formula is what nn.BatchNorm1d uses internally.

hidden_preact_fast = bngain * (hidden_prebn - hidden_prebn.mean(0, keepdim=True)) / torch.sqrt(hidden_prebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print(f'Max Diff: {(hidden_preact_fast - hidden_preact).abs().max()}')

# One-line backward through the whole BatchNorm block.
dhidden_prebn = bngain * bnvar_inv/batch_size * (batch_size * dhidden_preact - dhidden_preact.sum(0) - batch_size / (batch_size - 1) * bnraw * (dhidden_preact * bnraw).sum(0))
cmp('hidden_prebn: ', dhidden_prebn, hidden_prebn)

In [ ]:
# Putting it all together: full training using ONLY manual gradients (autograd disabled).
# The whole loop runs inside `with torch.no_grad()` so PyTorch never builds a backward graph.
# We compute forward, then apply our hand-derived gradients (one-go cross-entropy and one-go batchnorm)
# directly to the parameters. If our derivations are correct, training matches autograd's behavior.

n_embed = 10        # Dimensionality of the character embedding vectors
n_hidden = 200      # Number of neurons in the hidden layer of MLP
block_size = 3      # Context Length (# chars used to predict next)
seed = 2147483647   # For reproducibility between runs

# Initialize parameters from the given seed
gen = torch.Generator().manual_seed(seed)                   # Reproducibility
kaiming = (5 / 3) / ((n_embed * block_size)**0.5)           # Uniformity
lookup = torch.randn((vocab_size, n_embed),                 generator=gen)

# Hidden Layer 1
weights1 = torch.randn((n_embed * block_size, n_hidden),    generator=gen) * kaiming
biases1 = torch.randn(n_hidden,                             generator=gen) * .1

# Hidden Layer 2
weights2 = torch.randn((n_hidden, vocab_size),              generator=gen) * .1
biases2 = torch.randn(vocab_size,                           generator=gen) * .1

# Batch normalization parameters
bngain = torch.randn((1, n_hidden)) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden)) * 0.1


parameters = [lookup, weights1, biases1, weights2, biases2, bngain, bnbias]
print(f'Total Parameters: {sum(p.nelement() for p in parameters)}')

for p in parameters:
    p.requires_grad = True 

# Optimizations
max_steps = 200000
batch_size = 32
ix = torch.randint(0, Xtr.shape[0], (batch_size, ), generator=gen)
Xb, Yb = Xtr[ix], Ytr[ix]
lossi = []

# Use this context manager for efficiency: no autograd graph, no .grad bookkeeping.
with torch.no_grad():
 
    for i in range(max_steps):
        # Construct the mini batch
        ix = torch.randint(0, Xtr.shape[0], (batch_size, ), generator=gen)
        Xb, Yb = Xtr[ix], Ytr[ix]

        # Forward Pass
        embed = lookup[Xb]
        embedcat = embed.view(embed.shape[0], -1)

        # Linear Layer
        hidden_prebn = embedcat @ weights1 + biases1

        # Batch Normalization Layer
        bnmean = hidden_prebn.mean(0, keepdim=True)
        bnvar = hidden_prebn.var(0, keepdim=True, unbiased=True)
        bnvar_inv = (bnvar + 1e-5) ** -0.5
        bnraw = (hidden_prebn - bnmean) * bnvar_inv
        hidden_preact = bngain * bnraw + bnbias

        # Non-linearity
        hidden = torch.tanh(hidden_preact)
        logits = hidden @ weights2 + biases2
        loss = F.cross_entropy(logits, Yb)

        # Backward pass (manual, no autograd)
        for parameter in parameters:
            parameter.grad = None
        # loss.backward()

        # Manual Backpropagation: cross-entropy in one go
        dlogits = F.softmax(logits, 1)
        dlogits[range(batch_size), Yb] -= 1
        dlogits /= batch_size

        # 2nd Layer Backpropagation
        dhidden = dlogits @ weights2.T
        dweights2 = hidden.T @ dlogits
        dbiases2 = dlogits.sum(0)

        # Tanh Backpropagation
        dhidden_preact = (1.0 - hidden ** 2) * dhidden

        # Batch Normalization Backpropagation (one-go formula)
        dbngain = (bnraw * dhidden_preact).sum(0, keepdim=True)
        dbnbias = dhidden_preact.sum(0, keepdim=True)
        dhidden_prebn = bngain * bnvar_inv / batch_size * (batch_size * dhidden_preact - dhidden_preact.sum(0) - batch_size / (batch_size - 1) * bnraw * (dhidden_preact * bnraw).sum(0))

        # 1st Layer Backpropagation
        dembedcat = dhidden_prebn @ weights1.T
        dweights1 = embedcat.T @ dhidden_prebn
        dbiases1 = dhidden_prebn.sum(0)

        # Embedding Layer Backpropagation
        dembed = dembedcat.view(embed.shape)
        dlookup = torch.zeros_like(lookup)
        for j in range(Xb.shape[0]):
            for k in range(Xb.shape[1]):
                ix = Xb[j, k]
                dlookup[ix] += dembed[j, k]

        grads = [dlookup, dweights1, dbiases1, dweights2, dbiases2, dbngain, dbnbias]

        # Update with learning rate decay (using OUR gradients, not p.grad)
        lr = 0.1 if i < max_steps / 2 else 0.01
        for parameter, grad in zip(parameters, grads):
            parameter.data += -lr * grad

        # Track stats
        if i % 10000 == 0:
            print(f'{i:7d} / {max_steps:7d}: {loss.item():.4f}')
        lossi.append(loss.log10().item())

In [55]:
# Useful for checking gradients
for p, g in zip(parameters, grads):
    cmp(str(tuple(p.shape)), g, p)

TypeError: all() received an invalid combination of arguments - got (bool), but expected one of:
 * (Tensor input, *, Tensor out = None)
 * (Tensor input, tuple of ints dim = None, bool keepdim = False, *, Tensor out = None)
 * (Tensor input, int dim, bool keepdim = False, *, Tensor out = None)
 * (Tensor input, name dim, bool keepdim = False, *, Tensor out = None)


In [56]:
# Calibrate the batch normalization at the end of training

with torch.no_grad():
    # Pass the training set through
    embed = lookup[Xtr]
    embedcat = embed.view(embed.shape[0], -1)
    hiddenpre = embedcat @ weights1 + biases1

    # Measure the mean/std over the entire training set
    bnmean = hiddenpre.mean(0, keepdim=True)
    bnvar = hiddenpre.var(0, keepdim=True, unbiased=True)

In [57]:
# Evaluate the training Loss

@torch.no_grad() # Disabled gradient tracking
def split_loss(split):
    x,y = {
        'train': {Xtr, Ytr},
        'val': {Xdev, Ydev},
        'test': {Xte, Yte},
    }[split]
    embed = lookup[x] # N, block_size, n_embed
    embedcat = embed.view(embed.shape[0], -1) # concat into (N, block_size, n_embed)
    hidden_preact = (embedcat @ weights1) + biases1 # Hidden layer pre-activation
    
    # Batch Normalization
    hidden_preact = bngain * (hidden_preact - bnmean) * (bnvar + 1e-5) ** -0.5 + bnbias
    
    hidden = torch.tanh(hidden_preact) # (N, n_hidden)
    logits = hidden @ weights2 + biases2 # (N, vocab_size)
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.0730652809143066
val 2.112663745880127


In [58]:
# Sample from the model
gen = torch.Generator().manual_seed(seed + 10)

for _ in range(20):
    out = []
    context = [0] * block_size # Initialize with all
    
    while True:
        # Forward pass
        embed = lookup[torch.tensor([context])]
        hidden_preact = embed.view(embed.shape[0], -1) @ weights1 + biases1
        hidden_preact = bngain * (hidden_preact - bnmean) * (bnvar + 1e-5) ** -0.5 + bnbias
        hidden = torch.tanh(hidden_preact)
        logits = hidden @ weights2 + biases2
        probabilities = F.softmax(logits, dim=1)

        # Sample from the distribution
        ix = torch.multinomial(probabilities, num_samples=1, generator=gen).item()

        # Shift the context window and track the samples
        context = context[1:] + [ix]
        out.append(ix)

        # If we sample '.': break
        if ix == 0:
            break
    
    print(''.join(itos[i] for i in out)) # Decode and print the generated word

carla.
fate.
harli.
jari.
reity.
salaysie.
mahnee.
delynn.
jareei.
nellara.
chaiir.
kaleigh.
ham.
join.
quint.
shon.
marianni.
watero.
dearisi.
jace.
